In [38]:

import re
import csv
import time
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

try:
    from sentence_transformers import SentenceTransformer
    _MODEL = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
except Exception as _e:
    _MODEL = None
    print("[info] sentence-transformers/model available nahi:", type(_e).__name__,
          "-> embedding wale parts 'not run' honge\n")



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [39]:

class Retriever:
    def __init__(self, records, backend="tfidf"):
        if not records or any(not r["text"].strip() for r in records):
            raise ValueError("Provide nonempty records")
        self.records, self.backend = records, backend
        texts = [r["text"] for r in records]
        if backend == "tfidf":
            self.vectorizer = TfidfVectorizer()
            self.matrix = self.vectorizer.fit_transform(texts)
        elif backend == "embedding":
            if _MODEL is None:
                raise RuntimeError("embedding model not available")
            self.matrix = _MODEL.encode(texts, normalize_embeddings=True)
        else:
            raise ValueError("Use tfidf or embedding")

    def search(self, query, k=3):
        if not query.strip() or k <= 0:
            return []
        if self.backend == "tfidf":
            q = self.vectorizer.transform([query])
            if q.nnz == 0:
                return []
            scores = (q @ self.matrix.T).toarray()[0]
        else:
            q = _MODEL.encode([query], normalize_embeddings=True)[0]
            scores = self.matrix @ q
        ranked = sorted(range(len(self.records)), key=lambda i: (-float(scores[i]), i))
        return [(self.records[i], float(scores[i])) for i in ranked[:k]]



In [40]:

#                   UMAIR           TASK 1

def task1():
    print("=" * 70)
    print("TASK 1: Tokenization + FAQ chatbot")
    print("=" * 70)

    samples = ["AI, search!",
               "I can't log in.",
               "help@example.com",
               "CS4944 version 2.1",
               "",]

    for text in samples:
        ws = text.split()
        wl = re.findall(r"\b\w+\b", text, flags=re.UNICODE)
        print("Original  :", repr(text))
        print("Whitespace:", ws, len(ws))
        print("Word-like :", wl, len(wl))
        print()
    if _MODEL is not None:
        for text in [samples[1], samples[3]]:
            toks = _MODEL.tokenizer.tokenize(text)
            print("Model sub-words:", repr(text), toks, len(toks))
    else:
        print("Model tokenizer: not run (model available nahi)")

    print("\nDifference 1: split() 'can't' ko ek rakhta hai, regex 'can' + 't' bana deta hai.")
    print("Difference 2: regex email ko 3 hisson mein aur '2.1' ko '2','1' mein todta hai.")
    print("Normalization risk: lowercase/punctuation hatane se negation, numbers aur "
          "identifiers kharab ho sakte hain, isliye original text alag rakho.\n")


    FAQ = [ {"id": "faq:password", "question": "How do I reset my password?",
         "answer": "Use the recovery link on the login page.", "source": "it_help.txt"},
        {"id": "faq:submit", "question": "Where do I submit my assignment?",
         "answer": "Use the assignment upload page in the learning portal.",
         "source": "assessment.txt"},
        {"id": "faq:late", "question": "Can illness justify a late submission?",
         "answer": "Request an extension with medical evidence before the deadline.",
         "source": "handbook.txt"},

        {"id": "faq:wifi", "question": "How do I connect to the campus Wi-Fi?",
         "answer": "Select the campus network and sign in with your student account.",
         "source": "it_help.txt"},
        {"id": "faq:library", "question": "How long can I borrow a library book?",
         "answer": "Books can be borrowed for fourteen days and renewed once online.",
         "source": "library_rules.txt"},]
    records = [dict(r, text=r["question"] + " " + r["answer"]) for r in FAQ]
    engine = Retriever(records, "tfidf")

    def reply(query):
        if not query.strip():
            return "Please enter a question."
        for r in FAQ:
            if query.strip().casefold() == r["question"].casefold():
                return r["answer"] + " [source: " + r["source"] + "]"
        hits = engine.search(query)
        if not hits:
            return "No matching FAQ. Please contact the help desk."
        lines = ["Possible matches; choose or rephrase your question:"]
        for r, s in hits:
            lines.append(r["question"] + f" (similarity {s:.3f})")
        return "\n".join(lines)

    print("--- FAQ chatbot transcript ---")
    for q in ["How do I reset my password?",
              "password recovery",
              "",
              "volcano geology"]:
        print("You:", q)
        print("Bot:", reply(q))
        print()
    print("Confirmation kyun: similarity score probability ya sahi hone ka saboot nahi "
          "hota, isliye sirf exact question ka answer diya jata hai.\n")
    return reply


In [41]:

#               UMAIR
               TASK 3

def evaluate(scores, chunks, items):
    hits, rr = [], []
    for row, q in zip(scores, items):
        if q["doc"] is None:
            continue
        top = np.argsort(-row, kind="stable")[:3]
        good = [rank for rank, i in enumerate(top, 1)
                if chunks[i]["doc"] == q["doc"] and q["evidence"] in chunks[i]["text"]]
        hits.append(bool(good))
        rr.append(1 / good[0] if good else 0)
    return float(np.mean(hits)), float(np.mean(rr))

def task3():
    print("=" * 70)
    print("TASK 3: Retrieval evaluation")
    print("=" * 70)
    for q in ITEMS:
        if q["doc"] is not None:
            assert q["evidence"] in DOCS[q["doc"]]

    query_texts = [q["query"] for q in ITEMS]
    logs = []
    for strategy in ["0", "10", "sentence"]:
        chunks = make_chunks(DOCS, strategy)
        texts = [c["text"] for c in chunks]

        if _MODEL is not None:
            lens = [len(_MODEL.tokenizer(t, truncation=False)["input_ids"])
                    for t in texts + query_texts]
            print(f"[{strategy}] token audit: max={max(lens)} limit={_MODEL.max_seq_length}")
            if max(lens) > _MODEL.max_seq_length:
                raise ValueError("Model limit exceeded; rechunk")

        for method in ["tfidf", "embedding"]:
            if method == "embedding" and _MODEL is None:
                print(f"{strategy:8} embedding  -> not run (model available nahi)")
                continue
            t = time.perf_counter()
            if method == "tfidf":
                vec = TfidfVectorizer()
                C = vec.fit_transform(texts)
                Q = vec.transform(query_texts)
            else:
                C = _MODEL.encode(texts, normalize_embeddings=True)
                Q = _MODEL.encode(query_texts, normalize_embeddings=True)
            enc = time.perf_counter() - t
            t = time.perf_counter()
            scores = (Q @ C.T).toarray() if method == "tfidf" else Q @ C.T
            srch = time.perf_counter() - t

            for split in ["dev", "test"]:
                idx = [i for i, q in enumerate(ITEMS) if q["split"] == split]
                hit, mrr = evaluate(scores[idx], chunks, [ITEMS[i] for i in idx])
                print(f"{strategy:8} {method:9} {split:4} Hit@3={hit:.3f} MRR@3={mrr:.3f} "
                      f"chunks={len(chunks)} encode={enc:.4f}s search={srch:.5f}s")

            for row, q in zip(scores, ITEMS):
                for rank, i in enumerate(np.argsort(-row, kind="stable")[:3], 1):
                    c = chunks[i]
                    logs.append(dict(strategy=strategy, method=method, query=q["query"],
                                     split=q["split"], rank=rank, score=round(float(row[i]), 4),
                                     chunk_id=c["id"], doc=c["doc"], start=c["start"],
                                     end=c["end"], answerable=q["doc"] is not None,
                                     text=c["text"]))

    with open("retrieval_log.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(logs[0]))
        w.writeheader()
        w.writerows(logs)
    print("\nretrieval_log.csv ban gayi (top-3 text, score, ID, offsets)")
    print("\nFailure / edge cases (TF-IDF, sentence chunks):")
    chunks = make_chunks(DOCS, "sentence")
    eng = Retriever(chunks, "tfidf")
    for q in ["What proof is needed when I was ill?",
              "How should I hand in my coursework?",
              "What are tomorrow's flight prices?",
              "Who won the football match?"]:
        hits = eng.search(q, k=1)
        if hits:
            r, s = hits[0]
            print(f"  {q!r} -> {r['id']} (score {s:.3f}) chars {r['start']}-{r['end']}")
    print("Unanswerable query ko bhi text milta hai kyunki ranking hamesha sabse high score "
          "wala chunk deta hai; score sahi hone ka saboot nahi hai.")
    print("Selected strategy: sentence-aware (dev par best MRR, duplicate text nahi). "
          "Test result retune kiye baghair report karein.\n")



In [42]:

def document_chatbot():
    backend = "embedding" if _MODEL is not None else "tfidf"
    engine = Retriever(make_chunks(DOCS, "sentence"), backend)
    print(f"Evidence assistant ({backend}). Type exit to stop.")
    while True:
        try:
            query = input("Question: ")
        except (EOFError, KeyboardInterrupt):
            break
        if query.strip().lower() == "exit":
            break
        hits = engine.search(query)
        if not hits:
            print("No candidate evidence; rephrase or contact a human.")
            continue
        print("Candidate evidence; relevance must be checked:")
        for n, (r, s) in enumerate(hits, 1):
            print(n, r["id"], f"{s:.3f}", "characters", r["start"], r["end"])
        chosen = input("Choose 1-3 to inspect; Enter if none apply: ")
        if not chosen.isdigit() or not 1 <= int(chosen) <= len(hits):
            print("No answer selected. Rephrase or contact a human.")
            continue
        r, s = hits[int(chosen) - 1]
        print("Retrieved passage:", r["text"])
        print("Source:", r["doc"], "characters", r["start"], r["end"])


if __name__ == "__main__":
    import sys
    task1()
    task2()
    task3()
    if len(sys.argv) > 1 and sys.argv[1] == "chat":
        document_chatbot()


TASK 1: Tokenization + FAQ chatbot
Original  : 'AI, search!'
Whitespace: ['AI,', 'search!'] 2
Word-like : ['AI', 'search'] 2

Original  : "I can't log in."
Whitespace: ['I', "can't", 'log', 'in.'] 4
Word-like : ['I', 'can', 't', 'log', 'in'] 5

Original  : 'help@example.com'
Whitespace: ['help@example.com'] 1
Word-like : ['help', 'example', 'com'] 3

Original  : 'CS4944 version 2.1'
Whitespace: ['CS4944', 'version', '2.1'] 3
Word-like : ['CS4944', 'version', '2', '1'] 4

Original  : ''
Whitespace: [] 0
Word-like : [] 0

Model sub-words: "I can't log in." ['i', 'can', "'", 't', 'log', 'in', '.'] 7
Model sub-words: 'CS4944 version 2.1' ['cs', '##49', '##44', 'version', '2', '.', '1'] 7

Difference 1: split() 'can't' ko ek rakhta hai, regex 'can' + 't' bana deta hai.
Difference 2: regex email ko 3 hisson mein aur '2.1' ko '2','1' mein todta hai.
Normalization risk: lowercase/punctuation hatane se negation, numbers aur identifiers kharab ho sakte hain, isliye original text alag rakho.

---